<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 9 (fine-tuned DINOv2, alone and with YOLOv8s-cls)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 30 to 40 min).

Run 8's frozen-feature screen put DINOv2 ViT-S/14 first (out-of-fold ROC AUC 0.905, already above the fine-tuned single YOLOv8s-cls at 0.894), and its best partner was YOLOv8s-cls (pair 0.913, prediction correlation 0.83). Run 9 fine-tunes DINOv2 and compares three deployable options on the **same 5 folds** as Runs 6 to 8:

| Option | What is deployed | Size |
| :-- | :-- | :-- |
| D1 | DINOv2 ViT-S/14, fine-tuned, weights stored in half precision | about 44 MB |
| D2 | D1 with its linear layers stored as 8-bit integers (dynamic int8 quantization, run on the CPU) | about 23 MB |
| D3 | D2 averaged with one YOLOv8s-cls model (Run 7 settings, seed 42) | about 33 MB |

The bar is the Run 7 final model, the 3-seed YOLOv8s-cls average: out-of-fold macro F1 **0.801** at its recall-floor cutoff (ROC AUC 0.912). Out-of-fold scores for D2 and D3 come from the quantized model itself, so they are what would actually be deployed.

**The test set is used only if an option beats Run 7 in cross-validation.** If none does, the notebook stops after section 4 and Run 7 stays the final model. If one does, it is trained on all 761 images with 3 seeds and evaluated once on the held-out test set and on the external BU + BHD set, with size and CPU latency, exactly as in Run 7.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, time, json, random, math, re, zipfile, io, warnings
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, roc_curve, confusion_matrix
warnings.filterwarnings("ignore", message=".*quantiz.*"); warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = 42
FOLDS = 5
EPOCHS = 20                  # YOLOv8s-cls, Run 7 settings
DINO_EPOCHS = 15
BATCH = 32
IMGSZ = 224
SEEDS = [42, 43, 44]
RECALL_MIN = 0.90
MAX_MB, MAX_MS = 50, 200
NEAR_DUP_T = 5
RUN7_OOF_MACRO_F1, RUN7_OOF_AUC = 0.801, 0.912     # Run 7 final model (3-seed YOLOv8s-cls average), same folds
DATA_SRC = None
EXT_SRC = None
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"
YOLO_AUG = dict(dropout=0.3, auto_augment=None, erasing=0.0, scale=0.25, fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.2, hsv_v=0.1)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
OUT = os.path.join(BASE, "results", "run9"); os.makedirs(OUT, exist_ok=True)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
T_START = time.time()

## 1. Data: the cleaned split and the Run 6 folds

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)

splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
assert int(df["split"].notna().sum()) == len(splits1), "splits.csv did not match the download"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = set(n_lab[n_lab > 1].index)
clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
rank = {"train": 0, "val": 1, "test": 2}
drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in zip(ii, jj) if sp[i] != sp[j] and lb[i] == lb[j]}
drop |= {k for i, j in zip(ii, jj) if lb[i] != lb[j] for k in (i, j)}
clean = clean.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Removed {len(conflict_hashes)} conflicting-label groups and {len(drop)} near-duplicates")
print(pd.crosstab(clean["split"], clean["label"], margins=True))

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

POOL = os.path.join(BASE, "coral_pool"); shutil.rmtree(POOL, ignore_errors=True)
imgs = []
for _, r in clean.iterrows():
    d = os.path.join(POOL, r["label"]); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg")
    pad_square(r["path"]).save(dst, quality=95); imgs.append(dst)
clean["img"] = imgs
assert clean["img"].nunique() == len(clean), "file name clash in the image pool"

trval = clean[clean["split"] != "test"].reset_index(drop=True)
test = clean[clean["split"] == "test"].reset_index(drop=True)
y_trval = (trval["label"] == "bleached").astype(int).to_numpy()
y_test = (test["label"] == "bleached").astype(int).to_numpy()
test_files = test["img"].tolist()
FOLD_IDX = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(trval)), y_trval))
fold_of = np.zeros(len(trval), dtype=int)
for k, (_, va) in enumerate(FOLD_IDX): fold_of[va] = k
trval.assign(fold=fold_of)[["file", "label", "split", "fold"]].to_csv(os.path.join(OUT, "folds.csv"), index=False)
test[["file", "label"]].to_csv(os.path.join(OUT, "test_images.csv"), index=False)
print(f"Train + validation: {len(trval)} ({y_trval.sum()} bleached) in {FOLDS} folds | test: {len(test)} ({y_test.sum()} bleached)")

## 2. Helpers

YOLOv8s-cls training is the Run 7 code; DINOv2 fine-tuning is the Run 6 code (linear layer on the class token and mean patch token, dropout 0.3, AdamW with backbone 1e-5 and head 1e-3, cosine schedule, mixed precision, flips and crops). `quantize` converts a trained DINOv2 model to 8-bit linear layers for the CPU.

In [ ]:
def metrics_at(y, p, thr):
    yp = (np.asarray(p) >= thr).astype(int)
    return dict(macro_f1=f1_score(y, yp, average="macro"), bleached_recall=recall_score(y, yp, zero_division=0),
                bleached_precision=precision_score(y, yp, zero_division=0), healthy_recall=recall_score(y, yp, pos_label=0, zero_division=0),
                accuracy=accuracy_score(y, yp), roc_auc=roc_auc_score(y, p), cm=confusion_matrix(y, yp, labels=[0, 1]).tolist())

def pick_threshold(y, p):
    rows = [(t, f1_score(y, p >= t, average="macro"), recall_score(y, p >= t, zero_division=0)) for t in np.round(np.linspace(0.01, 0.99, 981), 3)]
    ok = [r for r in rows if r[2] >= RECALL_MIN]
    return max(ok or rows, key=lambda r: (r[1], r[0]))[0], bool(ok)

def make_folder(name, train_idx, val_idx):
    root = os.path.join(BASE, "cv", name); shutil.rmtree(root, ignore_errors=True)
    for part, idx in (("train", train_idx), ("val", val_idx)):
        for i in idx:
            r = trval.iloc[i]; d = os.path.join(root, part, r["label"]); os.makedirs(d, exist_ok=True)
            os.symlink(r["img"], os.path.join(d, os.path.basename(r["img"])))
    return root

from ultralytics import YOLO
def yolo_fit(data_root, name, seed):
    m = YOLO("yolov8s-cls.pt")
    m.train(data=data_root, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True, patience=EPOCHS + 1,
            val=False, plots=False, project=os.path.join(BASE, "runs7"), name=name, exist_ok=True, device=DEVICE, verbose=False, **YOLO_AUG)
    return str(m.trainer.last)

def yolo_predict(weights, files, bs=64):
    m = YOLO(weights); b = next(k for k, n in m.names.items() if n == "bleached"); p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=IMGSZ, device=DEVICE, verbose=False)]
    return np.array(p)

IMNET_MEAN, IMNET_STD = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

def load_dino_backbone():
    return torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)

class DinoClassifier(nn.Module):
    def __init__(self, backbone, n_classes=2, p=0.3):
        super().__init__()
        self.backbone, self.drop = backbone, nn.Dropout(p)
        self.head = nn.Linear(2 * backbone.embed_dim, n_classes)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.head(self.drop(torch.cat([f["x_norm_clstoken"], f["x_norm_patchtokens"].mean(1)], 1)))

_tensors = {}
def load_tensor(files, size):
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1)
                        for p in files])

def trval_tensor(size):
    if size not in _tensors: _tensors[size] = load_tensor(trval["img"].tolist(), size)
    return _tensors[size]

def norm(xb):
    return (xb.float() / 255 - IMNET_MEAN.to(xb.device)[:, None, None]) / IMNET_STD.to(xb.device)[:, None, None]

def augment(xb):
    if random.random() < 0.5: xb = xb.flip(3)
    if random.random() < 0.5: xb = xb.flip(2)
    n = xb.shape[-1]; c = int(n * random.uniform(0.87, 1.0)); i, j = random.randint(0, n - c), random.randint(0, n - c)
    return F.interpolate(xb[..., i:i + c, j:j + c], size=(n, n), mode="bilinear", align_corners=False)

def dino_fit(train_idx, seed, size):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    X, y = trval_tensor(size)[train_idx], torch.as_tensor(y_trval[train_idx])
    net = DinoClassifier(load_dino_backbone()).to(TDEV)
    opt = torch.optim.AdamW([{"params": net.backbone.parameters(), "lr": 1e-5}, {"params": net.head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    steps_per = math.ceil(len(X) / BATCH); total = DINO_EPOCHS * steps_per
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps_per) * 0.5 * (1 + math.cos(math.pi * s / total)))
    use_amp = TDEV.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    for ep in range(DINO_EPOCHS):
        net.train(); perm = torch.randperm(len(X))
        for k in range(0, len(perm), BATCH):
            idx = perm[k:k + BATCH]
            if len(idx) < 2: continue
            xb = augment(norm(X[idx].to(TDEV)))
            with torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(net(xb), y[idx].to(TDEV))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return net.eval()

def dino_predict(net, X, bs=64):
    out = []
    with torch.no_grad(), torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=TDEV.type == "cuda"):
        for k in range(0, len(X), bs):
            out.append(net(norm(X[k:k + bs].to(TDEV))).float().softmax(1)[:, 1].cpu())
    return torch.cat(out).numpy()

torch.backends.quantized.engine = "x86" if "x86" in torch.backends.quantized.supported_engines else torch.backends.quantized.supported_engines[-1]

def quantize(net):
    """8-bit dynamic quantization of the linear layers; runs on the CPU."""
    cpu = DinoClassifier(load_dino_backbone()).eval()
    cpu.load_state_dict({k: v.detach().float().cpu() for k, v in net.state_dict().items()})
    return torch.ao.quantization.quantize_dynamic(cpu, {nn.Linear}, dtype=torch.qint8).eval()

def cpu_predict(net, X, bs=16):
    with torch.no_grad():
        return torch.cat([net(norm(X[k:k + bs])).softmax(1)[:, 1] for k in range(0, len(X), bs)]).numpy()

def state_mb(net):
    b = io.BytesIO(); torch.save(net.state_dict(), b); return b.tell() / 1e6

## 3. Cross-validation: YOLOv8s-cls (seed 42), D1, D2, D3

The YOLOv8s-cls fits repeat Run 7's configuration A on the same folds and seed, so their scores should match Run 7. Progress prints after each fold.

In [ ]:
FOLD_ROOTS = [make_folder(f"fold{k}", tr, va) for k, (tr, va) in enumerate(FOLD_IDX)]
X224 = trval_tensor(IMGSZ)
oof = {n: np.full(len(trval), np.nan) for n in ("YOLOv8s-cls_s42", "D1_dinov2", "D2_dinov2_int8", "D3_dinov2_int8_plus_yolo")}
t0 = time.time()
for k, (tr, va) in enumerate(FOLD_IDX):
    oof["YOLOv8s-cls_s42"][va] = yolo_predict(yolo_fit(FOLD_ROOTS[k], f"Y_f{k}_s{SEED}", SEED), trval["img"].iloc[va].tolist())
    net = dino_fit(tr, SEED, IMGSZ)
    oof["D1_dinov2"][va] = dino_predict(net, X224[va])
    q = quantize(net); oof["D2_dinov2_int8"][va] = cpu_predict(q, X224[va])
    oof["D3_dinov2_int8_plus_yolo"][va] = (oof["D2_dinov2_int8"][va] + oof["YOLOv8s-cls_s42"][va]) / 2
    del net, q; torch.cuda.empty_cache()
    print(f">>> fold {k + 1}/{FOLDS}: ROC AUC " + ", ".join(f"{n} {roc_auc_score(y_trval[va], p[va]):.3f}" for n, p in oof.items())
          + f" ({(time.time() - t0) / 60:.1f} min)")
pd.DataFrame({"file": trval["file"], "true_label": y_trval, **oof}).to_csv(os.path.join(OUT, "oof_predictions.csv"), index=False)

## 4. Cross-validation results and the decision

In [ ]:
rows = []
for n, p in oof.items():
    thr, reached = pick_threshold(y_trval, p); mt = metrics_at(y_trval, p, thr)
    rows.append(dict(config=n, oof_roc_auc=roc_auc_score(y_trval, p), macro_f1_at_0_5=f1_score(y_trval, p >= 0.5, average="macro"),
                     cutoff=thr, recall_floor_reached=reached, macro_f1_at_cutoff=mt["macro_f1"], recall_at_cutoff=mt["bleached_recall"],
                     healthy_recall_at_cutoff=mt["healthy_recall"]))
rows.append(dict(config="Run 7 final (3-seed YOLOv8s-cls average)", oof_roc_auc=RUN7_OOF_AUC, macro_f1_at_cutoff=RUN7_OOF_MACRO_F1, recall_floor_reached=True))
cv = pd.DataFrame(rows); cv.to_csv(os.path.join(OUT, "cv_results.csv"), index=False)
print(cv.round(3).to_string(index=False))

cands = cv[cv["config"].str.startswith("D") & cv["recall_floor_reached"]].sort_values(["macro_f1_at_cutoff", "oof_roc_auc"], ascending=False)
BEST = cands.iloc[0]
CHANGE = bool(BEST["macro_f1_at_cutoff"] > RUN7_OOF_MACRO_F1)
FINAL, THR = BEST["config"], float(BEST["cutoff"])
print(f"\nBest Run 9 option: {FINAL} (out-of-fold macro F1 {BEST['macro_f1_at_cutoff']:.3f} vs Run 7 {RUN7_OOF_MACRO_F1:.3f}).")
print("It beats Run 7: the notebook continues to the final models and the test set." if CHANGE else
      "It does not beat Run 7: Run 7 stays the final model and the test set is NOT used. The remaining cells only write the summary.")

fig, ax = plt.subplots(figsize=(5.2, 4.6)); markers = ["o", "s", "^", "D"]
for i, (n, p) in enumerate(oof.items()):
    fpr, tpr, _ = roc_curve(y_trval, p)
    ax.plot(fpr, tpr, color=COLORS[i], lw=2, marker=markers[i], markevery=0.1, ms=5, label=f"{n} (AUC {roc_auc_score(y_trval, p):.3f})")
ax.plot([0, 1], [0, 1], color="#888888", lw=1, ls="--"); ax.axhline(RECALL_MIN, color="#888888", lw=1, ls=":")
ax.set_xlabel("False positive rate (healthy flagged as bleached)"); ax.set_ylabel("Bleached recall")
ax.set_title(f"Run 9 out-of-fold ROC, {FOLDS}-fold CV (n={len(trval)})", fontsize=10)
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "cv_roc.png"), dpi=150); plt.show()

## 5. Final models (only if a Run 9 option beat Run 7): all 761 images, 3 seeds

In [ ]:
FINALS = {}
if CHANGE:
    FULL_ROOT = make_folder("full", np.arange(len(trval)), FOLD_IDX[0][1])     # the val folder only satisfies Ultralytics
    for s in SEEDS:
        t0 = time.time(); m = {}
        net = dino_fit(np.arange(len(trval)), s, IMGSZ)
        if FINAL == "D1_dinov2":
            m["dino"] = net; m["path"] = os.path.join(OUT, f"final_dinov2_s{s}.pt"); torch.save({k: v.half() for k, v in net.state_dict().items()}, m["path"])
        else:
            m["dino_int8"] = quantize(net); m["path"] = os.path.join(OUT, f"final_dinov2_int8_s{s}.pt"); torch.save(m["dino_int8"].state_dict(), m["path"])
            del net
        if FINAL == "D3_dinov2_int8_plus_yolo":
            m["yolo"] = os.path.join(OUT, f"final_yolov8s_s{s}.pt"); shutil.copy(yolo_fit(FULL_ROOT, f"final_Y_s{s}", s), m["yolo"])
        FINALS[s] = m; torch.cuda.empty_cache()
        print(f">>> final seed {s}: {(time.time() - t0) / 60:.1f} min")

def predict_final(m, files):
    X = load_tensor(files, IMGSZ)
    if "dino" in m: return dino_predict(m["dino"], X)
    p = cpu_predict(m["dino_int8"], X)
    return (p + yolo_predict(m["yolo"], files)) / 2 if "yolo" in m else p

## 6. Held-out test set and external BU + BHD set (only if a Run 9 option beat Run 7)

In [ ]:
EXT = None
if CHANGE:
    p_seed = {s: predict_final(FINALS[s], test_files) for s in SEEDS}
    rows = []
    for s in SEEDS:
        rows += [dict(seed=s, at="cutoff", threshold=THR, **metrics_at(y_test, p_seed[s], THR)), dict(seed=s, at="0.5", threshold=0.5, **metrics_at(y_test, p_seed[s], 0.5))]
    test_res = pd.DataFrame(rows); test_res.to_csv(os.path.join(OUT, "test_results.csv"), index=False)
    pd.DataFrame(dict(file=test["file"], true_label=y_test, **{f"p_seed{s}": p_seed[s] for s in SEEDS})).to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
    print(test_res.drop(columns=["cm"]).round(3).to_string(index=False))
    TEST = {at: {c: (g[c].mean(), g[c].std(ddof=1)) for c in ["macro_f1", "bleached_recall", "healthy_recall", "accuracy", "roc_auc"]} for at, g in test_res.groupby("at")}
    for at, mm in TEST.items(): print(f"At {at}:", {k: f"{v[0]:.3f} ± {v[1]:.3f}" for k, v in mm.items()})

In [ ]:
if CHANGE:
    try:
        if EXT_SRC is None:
            from huggingface_hub import hf_hub_download
            EXT_SRC = hf_hub_download("esahit/coral-health-classification", "total_dataset.zip", repo_type="dataset")
        raw = os.path.join(BASE, "ext_raw"); shutil.rmtree(raw, ignore_errors=True)
        with zipfile.ZipFile(EXT_SRC) as z: z.extractall(raw)
        def parse(name):
            s = name.lower()
            lab = "bleached" if "unhealthy" in s else "healthy" if "healthy" in s else "dead" if "dead" in s else None
            src = "BHD" if ("hbd" in s or "bhd" in s) else "BU" if re.search(r"(?<![a-z])bu(?![a-z])", s) else "EILAT" if "eilat" in s else None
            return lab, src
        files = sorted(p for p in glob.glob(os.path.join(raw, "**", "*"), recursive=True) if p.lower().endswith(exts + (".tif", ".tiff")))
        ext = pd.DataFrame([(p, *parse(os.path.basename(p))) for p in files], columns=["path", "label", "source"])
        print(pd.crosstab(ext["source"].fillna("unknown"), ext["label"].fillna("unknown")))
        ext = ext[ext["label"].isin(["healthy", "bleached"]) & ext["source"].isin(["BU", "BHD"])].reset_index(drop=True)
        ext["phash"] = ext["path"].map(phash_of); ext = ext.dropna(subset=["phash"])
        n_raw = len(ext); ext = ext.drop_duplicates("phash").reset_index(drop=True); n_within = n_raw - len(ext)
        overlap = hamming(ext["phash"], df["phash"]).min(1) <= NEAR_DUP_T
        ext = ext[~overlap].reset_index(drop=True)
        sq = os.path.join(BASE, "ext_sq"); shutil.rmtree(sq, ignore_errors=True); os.makedirs(sq)
        ext["img"] = [os.path.join(sq, f"{r.source}_{i:05d}.jpg") for i, r in enumerate(ext.itertuples())]
        for r in ext.itertuples(): pad_square(r.path).save(r.img, quality=95)
        y_ext = (ext["label"] == "bleached").astype(int).to_numpy(); src = ext["source"].to_numpy()
        pe = {s: predict_final(FINALS[s], ext["img"].tolist()) for s in SEEDS}; pe_ens = np.mean([pe[s] for s in SEEDS], axis=0)
        rows = []
        for name, mask in (("BU + BHD", np.ones(len(y_ext), bool)), ("BU", src == "BU"), ("BHD", src == "BHD")):
            for s in SEEDS:
                rows.append(dict(set=name, n=int(mask.sum()), model=f"seed {s}", threshold=THR, **metrics_at(y_ext[mask], pe[s][mask], THR)))
            rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average", threshold=THR, **metrics_at(y_ext[mask], pe_ens[mask], THR)))
            rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average @0.5", threshold=0.5, **metrics_at(y_ext[mask], pe_ens[mask], 0.5)))
        EXT = pd.DataFrame(rows); EXT.to_csv(os.path.join(OUT, "external_results.csv"), index=False)
        EXT_INFO = dict(n=len(y_ext), bleached=int(y_ext.sum()), healthy=int((y_ext == 0).sum()), removed_within=n_within, removed_overlap=int(overlap.sum()),
                        by_source=ext["source"].value_counts().to_dict())
        print(EXT_INFO); print(EXT.drop(columns=["cm"]).round(3).to_string(index=False))
    except Exception as e:
        print("External test skipped:", repr(e))

## 7. Size and CPU latency of the deployed model (seed 42)

Mean over 100 test images, batch 1, after 10 warm-ups, image loading included, on Colab's CPU. Repeat on a team laptop for the final report.

In [ ]:
if CHANGE:
    m42 = FINALS[SEEDS[0]]
    if FINAL == "D1_dinov2":
        cpu_d = DinoClassifier(load_dino_backbone()).eval(); cpu_d.load_state_dict({k: v.float() for k, v in torch.load(m42["path"]).items()})
    else:
        cpu_d = m42["dino_int8"]
    ycpu = YOLO(m42["yolo"]) if "yolo" in m42 else None
    def one(p):
        x = torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((IMGSZ, IMGSZ), Image.BILINEAR))).permute(2, 0, 1)[None]
        with torch.no_grad():
            pr = cpu_d(norm(x)).softmax(1)[0, 1].item()
        if ycpu is not None: ycpu.predict(p, imgsz=IMGSZ, device="cpu", verbose=False)
        return pr
    for p in test_files[:10]: one(p)
    t0 = time.perf_counter()
    for p in test_files[:100]: one(p)
    LAT_MS = (time.perf_counter() - t0) / len(test_files[:100]) * 1000
    SIZE_MB = os.path.getsize(m42["path"]) / 1e6 + (os.path.getsize(m42["yolo"]) / 1e6 if "yolo" in m42 else 0)
    card = dict(run=9, final=FINAL, files=[os.path.basename(m42["path"])] + ([os.path.basename(m42["yolo"])] if "yolo" in m42 else []),
                dino="DinoClassifier(torch.hub facebookresearch/dinov2 dinov2_vits14) + linear on [CLS, mean patch tokens], dropout 0.3",
                dino_storage="fp16 state dict" if FINAL == "D1_dinov2" else "torch.ao.quantization.quantize_dynamic({nn.Linear}, qint8) state dict; rebuild the model, quantize, then load",
                input="pad to square with black, resize to 224, RGB in [0,1], ImageNet mean/std (DINOv2); YOLO uses its own preprocessing",
                combine="mean of P(bleached) from DINOv2 and YOLOv8s-cls" if "yolo" in m42 else "single model",
                classes={"0": "healthy", "1": "bleached"}, threshold=THR, threshold_source=f"{FOLDS}-fold out-of-fold predictions (recall >= {RECALL_MIN})",
                size_mb=round(SIZE_MB, 1), cpu_latency_ms=round(LAT_MS))
    json.dump(card, open(os.path.join(OUT, "final_model.json"), "w"), indent=2)
    print(f"Deployed model ({FINAL}): {SIZE_MB:.1f} MB, {LAT_MS:.0f} ms/image on CPU")

## 8. Summary for the progress log

In [ ]:
f3 = lambda m: f"{m[0]:.3f} ± {m[1]:.3f}"
ok = lambda b: "met" if b else "NOT met"
L = [f"RUN 9 SUMMARY ({time.strftime('%b. %d, %Y')})",
     f"Data: cleaned split; train + validation {len(trval)} in the Run 6 folds; test {len(test)}.",
     f"Cross-validation (out-of-fold, n={len(trval)}):"]
for r in cv.itertuples():
    L.append(f"  {r.config}: ROC AUC {r.oof_roc_auc:.3f}" + (f"; macro F1 @0.5 {r.macro_f1_at_0_5:.3f}; cutoff {r.cutoff:.3f} -> macro F1 {r.macro_f1_at_cutoff:.3f}, "
             f"bleached recall {r.recall_at_cutoff:.3f}, healthy recall {r.healthy_recall_at_cutoff:.3f}" if not pd.isna(r.cutoff) else f"; macro F1 at its cutoff {r.macro_f1_at_cutoff:.3f}"))
L.append(f"Best Run 9 option: {FINAL}; beats Run 7 in cross-validation: {CHANGE}.")
if CHANGE:
    L += [f"Held-out test (n={len(test)}, evaluated once, seeds {SEEDS}): at cutoff {THR:.3f} macro F1 {f3(TEST['cutoff']['macro_f1'])}, bleached recall {f3(TEST['cutoff']['bleached_recall'])}, "
          f"healthy recall {f3(TEST['cutoff']['healthy_recall'])}, ROC AUC {f3(TEST['cutoff']['roc_auc'])}; at 0.5 macro F1 {f3(TEST['0.5']['macro_f1'])}, recall {f3(TEST['0.5']['bleached_recall'])}.",
          "  Confusion matrices at the cutoff: " + "; ".join(f"seed {r.seed} {r.cm}" for r in test_res[test_res["at"] == "cutoff"].itertuples())]
    if EXT is not None:
        for setname in ("BU + BHD", "BU", "BHD"):
            g = EXT[EXT["set"] == setname]; sd = g[g["model"].str.startswith("seed")]
            L.append(f"  External {setname} (n={int(g['n'].iloc[0])}), at cutoff: macro F1 {f3((sd['macro_f1'].mean(), sd['macro_f1'].std(ddof=1)))}, "
                     f"recall {f3((sd['bleached_recall'].mean(), sd['bleached_recall'].std(ddof=1)))}")
    L += [f"Deployed model (seed 42): {SIZE_MB:.1f} MB, {LAT_MS:.0f} ms/image (Colab CPU).",
          f"Targets on test (at the cutoff, mean of 3 seeds): macro F1 >= 0.90 {ok(TEST['cutoff']['macro_f1'][0] >= 0.90)}; bleached recall >= 0.90 {ok(TEST['cutoff']['bleached_recall'][0] >= 0.90)}; "
          f"size < 50 MB {ok(SIZE_MB < MAX_MB)}; latency < 200 ms {ok(LAT_MS < MAX_MS)}."]
else:
    L.append("Test set not used; Run 7 stays the final model.")
L.append(f"Environment: GPU {GPU_NAME}, torch {torch.__version__}. Total time {(time.time() - T_START) / 60:.1f} min.")
summary = "\n".join(L); print(summary); open(os.path.join(OUT, "run9_summary.txt"), "w").write(summary)
ZIP = shutil.make_archive(os.path.join(BASE, "run9_results"), "zip", OUT); print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass